# Move a BTwin graph between formats

One building graph, four representations: **JSON-LD**, **NetworkX**, **RDF/SPARQL**, and **Neo4j**.
This tutorial converts between all of them, checks what survives each hop, and shows where BTwin
0.5.3 loses information or refuses to convert at all.

It works on the graph built in
[`../01-create-a-btwin-graph/`](../01-create-a-btwin-graph/create-a-btwin-graph.ipynb) — a copy of
that notebook's JSON-LD output ships here as `input/office-block-a.json`, so this tutorial is
self-contained and you can run it on its own.

Each representation gets its own interactive HTML page in `output/`, so you can click through what
survived a conversion instead of taking the triple counts on faith.

| Representation | Good for | BTwin entry point |
|---|---|---|
| JSON-LD | interchange, archival, the on-disk source of truth | `Serialization.JSONLDByObjects` |
| NetworkX | traversal, degree/path algorithms, plotting | `NetworkX.ByJSONLD`, `NetworkX.ByJSON` |
| RDF | SPARQL queries, ontology alignment, federation | `NetworkX.ToRDF`, `RDF.ByJSONLD`, `RDF.ByTTL` |
| Neo4j | a persistent, shared, indexed store | `NetworkX.ToNEO4J` |

**Prerequisites**

```bash
pip install "btwin[rdf]" neo4j
```

The Neo4j section is skipped unless you point it at a live database; everything else runs offline.

In [1]:
from pathlib import Path
import json

import btwin
from btwin import NetworkX, RDF, SPARQL, GraphPlot

from IPython.display import IFrame

OUTPUT = Path("output")
OUTPUT.mkdir(exist_ok=True)
SOURCE = Path("input") / "office-block-a.json"

BASE = "https://example.org/officeblocka/"

print("BTwin", btwin.__version__)
print("source exists:", SOURCE.exists())

BTwin 0.5.7
source exists: True


One helper, used after each conversion: it writes the graph to a self-contained interactive page
and embeds it below the cell. Comparing two of these side by side is a faster way to see what a
conversion did than reading triple counts.

In [2]:
def show(graph, step, title, height=460):
    """Write the graph to output/step-<step>.html and embed it below the cell."""
    path = OUTPUT / f"step-{step}.html"
    GraphPlot.NetworkXByHTML(graph, savePath=str(path), title=title,
                             nodeGroupBy="type", edgeGroupBy="type")
    print(f"{title}: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges -> {path}")
    return IFrame(src=str(path), width="100%", height=height)

## 1. The starting point: JSON-LD

JSON-LD is BTwin's on-disk format. One `@context` mapping prefixes to IRIs, one `@graph` array
holding every object.

In [3]:
jsonld = json.loads(SOURCE.read_text(encoding="utf-8"))

print("top-level keys :", list(jsonld.keys()))
print("@graph entries :", len(jsonld["@graph"]))
print()
print("prefixes:")
for prefix, iri in sorted(jsonld["@context"].items()):
    print(f"   {prefix:<8} {iri}")

top-level keys : ['@context', '@graph']
@graph entries : 22

prefixes:
   bot      https://w3id.org/bot#
   brick    https://brickschema.org/schema/Brick#
   btwin    btwin#
   eko      http://energy.linkeddata.es/em-kpi/ontology#
   ifc      https://standards.buildingsmart.org/IFC/DEV/IFC4/ADD2_TC1/OWL#
   time     https://www.w3.org/TR/2022/CRD-owl-time-20221115#


In [4]:
# One object, as stored
building = next(o for o in jsonld["@graph"] if o["@id"] == "bld-01")

print(json.dumps(building, indent=2))

{
  "@id": "bld-01",
  "@type": "bot:Building",
  "relationships": {},
  "name": "Office Block A"
}


Note `btwin` maps to the relative IRI `btwin#` rather than an absolute URL. That is harmless
inside BTwin, but it shows up later as a `btwin#isDocumentOf` predicate instead of a properly
namespaced one.

## 2. JSON-LD to NetworkX

`NetworkX.ByJSONLD` reads the document straight into a `MultiDiGraph`. It validates as it goes and
prints a report unless you turn that off.

In [5]:
G, report = NetworkX.ByJSONLD(jsonld)

print()
print("report ok:", report["ok"])

✓ NetworkX validation passed (all node/edge types are valid).

report ok: True


`ByJSONLD` returns **two** things: the graph and the validation report. Pass `printReport=False`
to keep it quiet and read the report instead.

Before 0.5.5 it returned the graph alone despite being annotated
`-> Tuple[Any, Dict[str, Any]]`, so `G, report = ...` failed with
`ValueError: too many values to unpack` (unpacking a graph iterates its nodes). If you are
following along on an older release, that is the difference.

In [6]:
G, _ = NetworkX.ByJSONLD(jsonld, printReport=False)

print(type(G).__name__)
print(f"nodes: {G.number_of_nodes()}   edges: {G.number_of_edges()}")
print()
print("node attributes are flat:")
print(json.dumps(G.nodes["space-02"], indent=2))

MultiDiGraph
nodes: 22   edges: 22

node attributes are flat:
{
  "id": "space-02",
  "type": "bot:Space",
  "name": "Meeting Room"
}


In [7]:
show(G, "01-byjsonld", "Step 1 - straight from JSON-LD", height=520)

Step 1 - straight from JSON-LD: 22 nodes, 22 edges -> output\step-01-byjsonld.html


## 3. NetworkX to node-link JSON and back

`ToJSON` writes NetworkX's own node-link format. This is *not* JSON-LD — it is a lossless-ish dump
of the graph structure, useful for caching a built graph or handing it to a JavaScript renderer.

In [8]:
nodeLink = NetworkX.ToJSON(G, savePath=OUTPUT / "office-block-a-nx.json")

parsed = json.loads(nodeLink)
print("keys:", list(parsed.keys()))
print()
print("first edge:")
print(json.dumps(parsed["edges"][0], indent=2))

keys: ['directed', 'edges', 'graph', 'multigraph', 'nodes']

first edge:
{
  "key": 0,
  "objectType": "bot:Building",
  "source": "storey-00",
  "subjectType": "bot:Storey",
  "target": "bld-01",
  "type": "brick:hasLocation"
}


Each edge carries `subjectType` and `objectType` alongside `type`, so an edge is self-describing
even without looking up its endpoints.

In [9]:
G2 = NetworkX.ByJSON(OUTPUT / "office-block-a-nx.json")

print(f"round trip: {G2.number_of_nodes()} nodes, {G2.number_of_edges()} edges")
print("same node set   :", set(G.nodes) == set(G2.nodes))
print("same attributes :", all(G.nodes[n] == G2.nodes[n] for n in G.nodes))

round trip: 22 nodes, 22 edges
same node set   : True
same attributes : False


In [10]:
# What actually changed?
before, after = G.nodes["bld-01"], G2.nodes["bld-01"]

print("before:", before)
print("after :", after)
print()
print("lost:", set(before) - set(after))

before: {'id': 'bld-01', 'type': 'bot:Building', 'name': 'Office Block A'}
after : {'name': 'Office Block A', 'type': 'bot:Building'}

lost: {'id'}


In [11]:
show(G2, "02-nodelink", "Step 2 - after the node-link round trip", height=520)

Step 2 - after the node-link round trip: 22 nodes, 22 edges -> output\step-02-nodelink.html


Identical to step 1, as it should be. Only the redundant `id` **attribute** is gone: node-link format uses it as the node *key*, so it is
consumed on the way back in. The key itself is intact, and nothing else is lost.

## 4. NetworkX to RDF

`ToRDF` returns a **tuple** of `(rdflib.Graph, turtleString)`. Pass `baseIRI` or your nodes become
relative IRIs like `<bld-01>`, which most triple stores will not thank you for.

In [12]:
rdfGraph, turtle = NetworkX.ToRDF(G, savePath=OUTPUT / "office-block-a.ttl", baseIRI=BASE)

print("returned:", type(rdfGraph).__name__, "+", type(turtle).__name__)
print("triples :", len(rdfGraph))

returned: Graph + str
triples : 90


In [13]:
print(turtle[:1200])

@prefix bot: <https://w3id.org/bot#> .
@prefix brick: <https://brickschema.org/schema/Brick#> .
@prefix btwin: <btwin#> .
@prefix eko: <http://energy.linkeddata.es/em-kpi/ontology#> .
@prefix ifc: <https://standards.buildingsmart.org/IFC/DEV/IFC4/ADD2_TC1/OWL#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<https://example.org/officeblocka/doc-01> a btwin:Document ;
    rdfs:label "O&M manual, AHU-1" ;
    btwin:isDocumentOf <https://example.org/officeblocka/bld-01> .

<https://example.org/officeblocka/doc-02> a btwin:Document ;
    rdfs:label "Commissioning report, Meeting Room" ;
    btwin:isDocumentOf <https://example.org/officeblocka/space-02> .

<https://example.org/officeblocka/kpiset-2026Q1> a btwin:KPISet ;
    rdfs:label "Energy & comfort, 2026 Q1" ;
    eko:hasAssociatedObject <https://example.org/officeblocka/bld-01> .

<https://example.org/officeblocka/pt-c-01> a brick:CO2_Sensor ;
    rdfs:label "Meeting Room 

## 5. The other road to RDF: straight from JSON-LD

`RDF.ByJSONLD` skips NetworkX entirely. It also returns `(graph, turtle)`.

Strict mode — the default — checks every node and every relationship target as it goes. Targets
without an `@id` are treated as blank nodes when they carry a `@type`, which is how the KPI set's
`eko:hasEvaluationTimestep` time interval survives.

(Before 0.5.5 that interval was rejected outright, so strict mode refused documents BTwin itself
had just written and you had to pass `strict=False`.)

In [14]:
directGraph, directTurtle = RDF.ByJSONLD(
    jsonld, savePath=OUTPUT / "office-block-a-direct.ttl", baseIRI=BASE)

print(f"via NetworkX  : {len(rdfGraph)} triples")
print(f"direct        : {len(directGraph)} triples")
print(f"difference    : {len(directGraph) - len(rdfGraph)}")

via NetworkX  : 90 triples
direct        : 121 triples
difference    : 31


In [15]:
# The interval that used to be rejected, now carried through as a blank node
for line in directTurtle.splitlines():
    if "hasBeginning" in line or "hasEnd" in line or "interval" in line:
        print(line)

    eko:hasEvaluationTimestep [ a time1:interval ;
            time1:hasBeginning "2026-01-01T00:00:00Z" ;
            time1:hasEnd "2026-03-31T23:59:59Z" ] .


Sixteen more triples. Which ones?

Comparing the two graphs with `set(a) - set(b)` gives a misleading answer, because each conversion
mints fresh blank nodes for the IFC properties — every triple touching one looks "different" even
when it says the same thing. Counting **predicates** instead cuts through that:

In [16]:
from collections import Counter

viaNetworkX = Counter(RDF.Compact(rdfGraph, p) for _, p, _ in rdfGraph)
direct = Counter(RDF.Compact(directGraph, p) for _, p, _ in directGraph)

print(f"{'predicate':<26}{'via NX':>8}{'direct':>8}{'diff':>6}")
for predicate in sorted(set(viaNetworkX) | set(direct)):
    a, b = viaNetworkX.get(predicate, 0), direct.get(predicate, 0)
    flag = "  <--" if a != b else ""
    print(f"{predicate:<26}{a:>8}{b:>8}{b - a:>6}{flag}")

predicate                   via NX  direct  diff
brick:hasLocation               15      15     0
btwin#hasKPIs                    0       3     3  <--
btwin#isDocumentOf               2       2     0
eko:hasAssociatedObject          1       1     0
eko:hasEvaluationTimestep        0       1     1  <--
ifc:HasProperties                8       8     0
ifc:HasPropertySets              4       4     0
ifc:NominalValue                 8      11     3  <--
ifc:Unit                         0       7     7  <--
rdf:type                        22      34    12  <--
rdfs:label                      30      33     3  <--
time1:hasBeginning               0       1     1  <--
time1:hasEnd                     0       1     1  <--


The difference falls into three groups.

**The IFC properties.** The direct path emits 8 `rdf:type` triples
(`ifc:IfcPropertySingleValue`) and 4 `ifc:Unit` triples that the NetworkX path never writes. That
second one matters: through NetworkX, `netFloorArea` arrives as the bare number `28.5` with **no
unit attached** — the `sqm` is gone.

**The KPIs themselves.** `btwin#hasKPIs` (3), the `eko:KPI` types, and the `ifc:NominalValue` and
`ifc:Unit` on each: the direct path writes every KPI as its own node carrying its figure and its
unit. `AddNodeByObject` instead folds them into attributes of the KPI-set node — `energyUseIntensity:
41.2` sitting on `kpiset-2026Q1` — so by the time `ToRDF` runs there is no KPI node left to walk.

**The KPI evaluation interval.** `eko:hasEvaluationTimestep`, its `time:interval` type, and the
`hasBeginning` / `hasEnd` pair: four triples describing a blank node that never becomes a NetworkX
node, so `ToRDF` has nothing to write.

The first group is a setting, not a law: `AddNodeByObject` takes `keepPSetMetadata`, which keeps
the original `ifc:HasProperties` structure instead of only flattening it.

`ByJSONLD` does not expose that flag, so to use it you have to build the graph the long way — the
same two passes from the first tutorial:

In [17]:
detailed = NetworkX.Constructor("MultiDiGraph", name="Office Block A")

for obj in jsonld["@graph"]:
    NetworkX.AddNodeByObject(detailed, obj, keepPSetMetadata=True)
for obj in jsonld["@graph"]:
    NetworkX.AddEdgesByObject(detailed, obj)

detailedGraph, _ = NetworkX.ToRDF(detailed, baseIRI=BASE)

print(f"default ByJSONLD      : {len(rdfGraph)} triples")
print(f"keepPSetMetadata=True : {len(detailedGraph)} triples")
print(f"direct from JSON-LD   : {len(directGraph)} triples")
print()
print("units preserved:",
      sum(1 for _, p, _ in detailedGraph if RDF.Compact(detailedGraph, p) == "ifc:Unit"))
print("still missing   :", len(directGraph) - len(detailedGraph), "triples")

default ByJSONLD      : 90 triples
keepPSetMetadata=True : 102 triples
direct from JSON-LD   : 121 triples

units preserved: 4
still missing   : 19 triples


In [18]:
show(detailed, "03-detailed", "Step 3 - keepPSetMetadata=True", height=520)

Step 3 - keepPSetMetadata=True: 22 nodes, 22 edges -> output\step-03-detailed.html


Click a `pset-*` node in step 3 and compare it with step 1: the property structure is still there
rather than flattened away.

That recovers the units and closes all but 19 triples. What remains is everything to do with the
KPI set's contents, and no NetworkX route can carry any of it: the KPIs are flattened to attributes
rather than kept as nodes, and the evaluation interval is a blank node — not a node with an `@id` —
so it never enters the graph in the first place.

So the choice is:

- `ByJSONLD` — convenient, flattened, no units, no KPI figures, no interval.
- manual `AddNodeByObject(..., keepPSetMetadata=True)` — verbose, keeps the properties whole.
- `RDF.ByJSONLD` — complete, and skips NetworkX entirely.

## 6. Reading Turtle back

`RDF.ByTTL` loads a `.ttl` file into an `rdflib.Graph`. Unlike the two converters above, it returns
the graph on its own.

In [19]:
reloaded = RDF.ByTTL(OUTPUT / "office-block-a.ttl")

print(type(reloaded).__name__, "with", len(reloaded), "triples")
print("survived the file round trip:", len(reloaded) == len(rdfGraph))

Graph with 90 triples
survived the file round trip: True


## 7. Getting your bearings in an RDF graph

`RDF.Index` and `RDF.SchemaSummary` exist so you can look at an unfamiliar graph without writing
exploratory SPARQL first.

In [20]:
index = RDF.Index(rdfGraph)

print("index keys:", list(index.keys()))
print("subjects  :", len(index["nodes"]))
print()
print("predicate counts:")
for predicate, count in sorted(index["predicates"].items(), key=lambda kv: -kv[1]):
    print(f"   {count:>3}  {predicate}")

index keys: ['nodes', 'labelIndex', 'predicates']
subjects  : 22

predicate counts:
    30  rdfs:label
    22  rdf:type
    15  brick:hasLocation
     8  ifc:NominalValue
     8  ifc:HasProperties
     4  ifc:HasPropertySets
     2  btwin#isDocumentOf
     1  eko:hasAssociatedObject


In [21]:
# Labels are indexed too, and a shared label keeps every matching IRI
for label, iris in list(index["labelIndex"].items())[:5]:
    print(f"{label:<36} {iris}")

Office Block A                       ['https://example.org/officeblocka/bld-01']
O&M manual, AHU-1                    ['https://example.org/officeblocka/doc-01']
Commissioning report, Meeting Room   ['https://example.org/officeblocka/doc-02']
Energy & comfort, 2026 Q1            ['https://example.org/officeblocka/kpiset-2026Q1']
Pset_SpaceCommon                     ['https://example.org/officeblocka/pset-space-01', 'https://example.org/officeblocka/pset-space-02', 'https://example.org/officeblocka/pset-space-03', 'https://example.org/officeblocka/pset-space-04']


In [22]:
summary = RDF.SchemaSummary(rdfGraph, index=index)

print(summary["text"][:1400])

PREFIXES
  PREFIX bot: <https://w3id.org/bot#>
  PREFIX brick: <https://brickschema.org/schema/Brick#>
  PREFIX btwin: <btwin#>
  PREFIX eko: <http://energy.linkeddata.es/em-kpi/ontology#>
  PREFIX ifc: <https://standards.buildingsmart.org/IFC/DEV/IFC4/ADD2_TC1/OWL#>
  PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
  PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

CLASSES (used as rdf:type)
  bot:Building  - A building within a Site; contains one or more Storeys and Spaces.
  bot:Space  - A bounded part of the built environment (e.g., room, corridor).
  bot:Storey  - A level of a building (above/below ground) that contains Spaces.
  brick:CO2_Sensor
  brick:Energy_Sensor
  brick:Humidity_Sensor
  brick:Temperature_Sensor
  brick:Zone  - A logical grouping of spaces defined by a subsystem (e.g., HVAC, lighting, fire).
  btwin#Document
  btwin#KPISet
  ifc:IfcPropertySet  - The IfcPropertySet is a container that holds properties within a property tree. These properties a

`RDF.Compact` shortens an IRI using the graph's registered prefixes — handy when printing results.

In [23]:
for iri in ["https://brickschema.org/schema/Brick#CO2_Sensor",
            "https://w3id.org/bot#Space",
            BASE + "space-02"]:
    print(f"{RDF.Compact(rdfGraph, iri):<22} <- {iri}")

brick:CO2_Sensor       <- https://brickschema.org/schema/Brick#CO2_Sensor
bot:Space              <- https://w3id.org/bot#Space
https://example.org/officeblocka/space-02 <- https://example.org/officeblocka/space-02


The last one stays long: `BASE` is not a registered prefix, so there is nothing to compact against.

## 8. Querying with SPARQL

`SPARQL.Validate` checks a query before you run it. It returns `(cleanedQuery, message)` — the
cleaned query is `None` when the query is unusable, and `message` explains why. It also appends a
`LIMIT` if you left one off.

In [24]:
query = '''
PREFIX bot:   <https://w3id.org/bot#>
PREFIX brick: <https://brickschema.org/schema/Brick#>
PREFIX rdfs:  <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?sensor ?sensorName ?spaceName WHERE {
  ?sensor a brick:CO2_Sensor ;
          rdfs:label ?sensorName ;
          brick:hasLocation ?space .
  ?space  a bot:Space ;
          rdfs:label ?spaceName .
}
'''

cleaned, message = SPARQL.Validate(query)

print("usable  :", cleaned is not None)
print("message :", repr(message))
print()
print("cleaned query ends with:", cleaned.strip().splitlines()[-1])

usable  : True
message : ''

cleaned query ends with: LIMIT 100


In [25]:
rows = RDF.Query(rdfGraph, query)

for row in rows:
    print(f"{row['sensorName']:<20} in {row['spaceName']}")

Meeting Room CO2     in Meeting Room
Open Office CO2      in Open Office


In [26]:
# RDF.SourceNodes pulls the IRIs out of a result set - useful for chaining one query into the next
print(RDF.SourceNodes(rdfGraph, rows))

['https://example.org/officeblocka/pt-c-01', 'https://example.org/officeblocka/pt-c-02']


A query that crosses the whole model — from a document, up to what it describes, down to the
sensors in it.

The `VALUES` clause is doing real work here. `brick:hasLocation` is the *same* predicate for
"sensor is in a space" and "storey is in a building", so without pinning `?sensor` to an actual
sensor class the query happily reports Ground Floor as a sensor of Office Block A.

In [27]:
crossQuery = '''
PREFIX bot:   <https://w3id.org/bot#>
PREFIX brick: <https://brickschema.org/schema/Brick#>
PREFIX btwin: <btwin#>
PREFIX rdfs:  <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?docName ?subjectName ?sensorName WHERE {
  VALUES ?sensorType { brick:Temperature_Sensor brick:CO2_Sensor
                       brick:Humidity_Sensor brick:Current_Sensor }

  ?doc     a btwin:Document ;
           rdfs:label ?docName ;
           btwin:isDocumentOf ?subject .
  ?subject rdfs:label ?subjectName .
  ?sensor  a ?sensorType ;
           brick:hasLocation ?subject ;
           rdfs:label ?sensorName .
}
'''

for row in RDF.Query(rdfGraph, crossQuery):
    print(f"{row['docName']:<38} {row['subjectName']:<14} {row['sensorName']}")

Commissioning report, Meeting Room     Meeting Room   Meeting Room air temperature
Commissioning report, Meeting Room     Meeting Room   Meeting Room CO2


## 9. Neo4j

`NetworkX.ToNEO4J` pushes the graph into a Neo4j database. The mapping:

| BTwin | Neo4j | Note |
|---|---|---|
| node `type` | node label | `:` and non-word characters become `_`, so `bot:Space` becomes `bot_Space` |
| node key | `UID` property | falls back through `UID` -> `id` -> the node key |
| other node attributes | node properties | keys sanitized the same way |
| edge `type` | relationship type | `brick:hasLocation` becomes `brick_hasLocation` |

Property-set values ride along as node properties, so a space arrives in Neo4j already carrying
its `netFloorArea` and `occupancyType`.

It `MERGE`s rather than `CREATE`s, so re-running is idempotent. `wipeDb=True` clears the database
first — it runs `MATCH (n) DETACH DELETE n`, so never point that at a database you care about.

`NEO4J_URI` and `NEO4J_PASSWORD` are required and raise `ValueError` when missing. Until 0.5.5 they
defaulted to someone else's Aura instance and the literal string `'***'`.

In [28]:
import os

NEO4J_URI = os.environ.get("NEO4J_URI")
NEO4J_USERNAME = os.environ.get("NEO4J_USERNAME", "neo4j")
NEO4J_PASSWORD = os.environ.get("NEO4J_PASSWORD")

if NEO4J_URI and NEO4J_PASSWORD:
    stats = NetworkX.ToNEO4J(
        G,
        NEO4J_URI=NEO4J_URI,
        NEO4J_USERNAME=NEO4J_USERNAME,
        NEO4J_PASSWORD=NEO4J_PASSWORD,
        wipeDb=False,
    )
    print(stats)
else:
    print("Skipped: set NEO4J_URI and NEO4J_PASSWORD to run this cell.")
    print()
    print("A throwaway local instance:")
    print("   docker run --rm -p 7474:7474 -p 7687:7687 \\")
    print("       -e NEO4J_AUTH=neo4j/testtest neo4j:5")
    print()
    print("   export NEO4J_URI=bolt://localhost:7687")
    print("   export NEO4J_PASSWORD=testtest")

Skipped: set NEO4J_URI and NEO4J_PASSWORD to run this cell.

A throwaway local instance:
   docker run --rm -p 7474:7474 -p 7687:7687 \
       -e NEO4J_AUTH=neo4j/testtest neo4j:5

   export NEO4J_URI=bolt://localhost:7687
   export NEO4J_PASSWORD=testtest


Once it has run, the graph is browsable at <http://localhost:7474> with Cypher:

```cypher
// every sensor and the space it sits in
MATCH (sensor)-[:brick_hasLocation]->(space:bot_Space)
WHERE sensor.type STARTS WITH 'brick:'
RETURN sensor.UID, sensor.name, space.name

// the containment chain from a sensor up to the building
MATCH path = (s {UID: 'pt-c-01'})-[:brick_hasLocation*]->(b:bot_Building)
RETURN path
```

The property values that ride along come from `NodeLinkedPSets`, which reads a node's linked
property sets:

In [29]:
for psetUID, attrs in NetworkX.NodeLinkedPSets(nxGraph=G, nodeObjectUID="space-02"):
    print(f"{psetUID}:")
    for key in ("netFloorArea", "occupancyType"):
        print(f"   {key:<15} {attrs[key]}   -> a Neo4j node property")

pset-space-02:
   netFloorArea    28.5   -> a Neo4j node property
   occupancyType   meeting   -> a Neo4j node property


In 0.5.3 and 0.5.4 none of this reached Neo4j: the enrichment called
`NodeLinkedPSets(graph=..., nodeUID=...)` while the real parameters are
`(nxGraph, nodeObjectUID)`, and the resulting `TypeError` was swallowed by a bare
`except Exception`, so every node arrived without its properties and nothing said so.

You can also flatten the property sets yourself before exporting, which drops the separate PSet
nodes entirely rather than carrying them alongside:

In [30]:
flattened = NetworkX.CompactPSets(G.copy())

print("ready to export, with properties inlined:")
print(json.dumps(flattened.nodes["space-02"], indent=2))

{'psetsFound': 4, 'psetsCompacted': 4, 'psetsOrphanDeleted': 0, 'ownersTouched': 4, 'propertiesAttached': 8, 'conflicts': 0}
ready to export, with properties inlined:
{
  "id": "space-02",
  "type": "bot:Space",
  "name": "Meeting Room",
  "netFloorArea": 28.5,
  "occupancyType": "meeting"
}


In [31]:
show(flattened, "04-flattened", "Step 4 - property sets folded in, ready for Neo4j", height=520)

Step 4 - property sets folded in, ready for Neo4j: 18 nodes, 18 edges -> output\step-04-flattened.html


## 10. What survives each hop

Measured on this graph, not from the docs:

In [32]:
summaryRows = [
    ("JSON-LD (source)",          len(jsonld["@graph"]), "objects"),
    ("-> NetworkX",               G.number_of_nodes(),   "nodes"),
    ("-> node-link JSON -> back", G2.number_of_nodes(),  "nodes"),
    ("-> RDF via NetworkX",       len(rdfGraph),         "triples"),
    ("-> RDF, keepPSetMetadata",  len(detailedGraph),   "triples"),
    ("-> RDF direct from JSON-LD", len(directGraph),     "triples"),
    ("-> Turtle file -> back",    len(reloaded),         "triples"),
]

for label, count, unit in summaryRows:
    print(f"{label:<30} {count:>4} {unit}")

JSON-LD (source)                 22 objects
-> NetworkX                      22 nodes
-> node-link JSON -> back        22 nodes
-> RDF via NetworkX              90 triples
-> RDF, keepPSetMetadata        102 triples
-> RDF direct from JSON-LD      121 triples
-> Turtle file -> back           90 triples


Reading that table:

- **JSON-LD to NetworkX is 1:1 at the object level**, but nested structures — the KPIs inside the
  KPI set, the properties inside a property set — do not become nodes. They stay as attributes or
  are dropped. That is a fact about *this hop*, not about the data: `RDF.ByJSONLD` does make them
  nodes, which is most of why the two triple counts differ.
- **The node-link round trip is faithful**, minus the redundant `id` attribute.
- **The direct JSON-LD to RDF path is the most complete**, 121 triples against 90: it types each
  IFC property, keeps its unit, writes every KPI as its own node with its figure and unit, and
  carries the KPI evaluation interval.
- **Turtle is a true round trip.**

Pick the path by what you need to keep, not by which is shortest. If units matter downstream, do
not reach RDF through NetworkX.

## Questions to try

1. **What exactly are the four triples `detailedGraph` still lacks?** Compare it with
   `directGraph` by predicate, remembering that blank node ids differ between the two, so a plain
   set difference will mislead you.
2. **Write a SPARQL query that finds spaces with no sensor.** `FILTER NOT EXISTS` is the clause
   you want, and the `VALUES` trick from section 8 still applies. Check it against the picture from
   the first tutorial.
3. **Does `CompactPSets` change the RDF?** Run `ToRDF` on `flattened` and compare the triple count
   with the 90 from section 4. Where did the difference come from?
4. **Round trip through Neo4j.** With a database running, export, then read back with a Cypher
   query and rebuild a NetworkX graph from the result. What is missing compared to `G`?

## Where to go next

- [`../01-create-a-btwin-graph/`](../01-create-a-btwin-graph/create-a-btwin-graph.ipynb) — how the
  source graph was built.
- [`../03-llm-in-action/`](../03-llm-in-action/llm-in-action.ipynb) — build and query a graph with
  a language model.
- [`../00-early-adopters/`](../00-early-adopters/early-adopters.ipynb) — the method-by-method API reference.
- `Cycle` and `Tool` in `btwin.llm` — the same SPARQL machinery, driven by a model instead of by
  hand.